In [45]:
# pip install torch
from pathlib import Path
import json
import torch

# Colab Drive mount:
# from google.colab import drive
# drive.mount("/content/drive")

# relative to the kernel's working directory, change this to an absolute path if needed
MODEL_DIR = (Path.cwd() / "models" / "d30").resolve()
DATA_DIR = Path.cwd() / "data"
OUTPUT_DIR = Path.cwd() / "my_trained_model"

RUN_INFERENCE = True  # Set to False to skip inference and only run the training loop

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = torch.device("mps"), torch.float32
else:
    DEVICE, DTYPE = torch.device("cpu"), torch.float32
print("Model folder:", MODEL_DIR)
print("Device:", DEVICE, "Inference dtype:", DTYPE)

Model folder: /Users/pranav26/Desktop/Pranav/Personal_Projects/Barebones_LLM_implementation/models/d30
Device: mps Inference dtype: torch.float32


### BPE Tokenizer

In [46]:
import json
import re
from collections import Counter


class BPE:

    def __init__(self, filename=None):

        if filename is None:
            self.vocab = [chr(i) for i in range(256)]
            self.merges = {}
            self.special_tokens = {}
        else:
            with open(filename, "rt") as f:
                self.vocab, merge_keys, merge_values, self.special_tokens = json.load(f)
            self.special_tokens = {int(k):v for k, v in self.special_tokens.items()}
            self.merges = {tuple(k):v for k, v in zip(merge_keys, merge_values)}


    @staticmethod
    def merge_pair(word, a, b, merged):

        i = 0
        while i < len(word)-1:
            if word[i] == a and word[i+1] == b:
                word[i] = merged
                word.pop(i+1)
            i += 1
        

    def replace_special_tokens(self, word):

        for k,v in self.special_tokens.items():
            word = word.replace(v, chr(k))
        return word


    def train(self, text, target_vocab_size):

        word_freqs = Counter(re.findall(r"\s\S+|\s+|\S+", text)) # \s is whitespace, \S is non-whitespace character

        #we replace the special tokens in words before merging
        word_freqs = {self.replace_special_tokens(k):v for k, v in word_freqs.items() if v > 1} # only taking words with > 1 freq
        words = [[ord(c) for c in word if ord(c) not in self.special_tokens] for word in word_freqs] # each word is a list of the numerical repr of each char; we don't include special tokens in merging
        word_counts = word_freqs.values()

        for l in range(len(self.vocab), target_vocab_size):
            pair_freqs = Counter()
            for word, word_cnt in zip(words, word_counts):
                for i in range(len(word)-1):
                    pair_freqs[(word[i], word[i+1])] += word_cnt

            a,b = pair_freqs.most_common(1)[0][0]
            self.merges[(a,b)] = l # merge order is saved by the index in vocab
            self.vocab.append(self.vocab[a] + self.vocab[b])

            for word in words:
                self.merge_pair(word, a, b, l)

        # adding the special tokens to vocab
        self.vocab += [""]*(max(self.special_tokens.keys()) - len(self.vocab)+1)
        for k,v in self.special_tokens.items():
            self.vocab[k] = v

    def encode(self, text):

        tokens = []
        for m in re.finditer(r"\s\S+|\s+|\S+", text):
            word = [ord(c) for c in self.replace_special_tokens(m.group(0))] # replace the special tokens while encoding
            while True:
                word_merges = {}
                for pair in zip(word[:-1], word[1:]):
                    if pair in self.merges:
                        word_merges[pair] = self.merges[pair]

                if len(word_merges)==0:
                    break

                a, b = min(word_merges, key=word_merges.get)
                self.merge_pair(word, a, b, self.merges[(a,b)])

            tokens.extend(word)
        return tokens

    def decode(self, tokens):

        return "".join([self.vocab[t] for t in tokens if t < len(self.vocab)])

    def save(self, fname):

        # json doesn't store dicts with keys other than strings, so we save merges as lists of keys and values seperately
        with open(fname, "wt") as f:
            json.dump([self.vocab,
                       list(self.merges.keys()),
                       list(self.merges.values()),
                       self.special_tokens], f)



### LLM Architecture with RoPE and KV-caching


In [47]:
import torch
import math

def embedding(tokens, weight, dtype):

    return weight[tokens].to(dtype=dtype) # weights is a vocab_size*d dim vector, where d is embedding dim

def linear(x, weight):

    return x @ weight.type_as(x) # we represent the input x as 16b, while weights as 32b

def silu(x):

    # silu = sigmoidal linear unit
    return x/(1 + torch.exp(-x))

def rms_norm(x):

    x_ = x.float() # doing norm in 16bit could lead to loss of precision
    x_norm = (x_ / (x_**2).mean(dim=-1, keepdim=True).sqrt().clamp_min(1e-12)).type_as(x)   # basically divide each row of x by its magn, and then reset its type back to 16b
    return x_norm

def softmax(x):

    x = (x - x.max(dim=-1, keepdim=True).values).exp()
    return x / x.sum(dim=-1, keepdim=True)

def self_attn(q,k,v,mask):

    w = (q @ k.transpose(-1, -2)/ math.sqrt(q.shape[-1])) + mask.type_as(q)
    return softmax(w) @ v


class LLM:

    def rope(self, x, pos=0):

        L = x.shape[2]
        a = self.buffers["rope1"][pos:pos+L] * x
        b = self.buffers["rope2"][pos:pos+L] * x.reshape(-1, 2).flip(-1).reshape_as(x)
        return a+b


    def multihead_attn(self, x, layer, mask, pos=0, cache=None):

        B, L, d = x.shape # B: batch_size, L: seq_len, d: embed_dim
        q = linear(x, self.params[f"wq_{layer}"])
        k = linear(x, self.params[f"wk_{layer}"])
        v = linear(x, self.params[f"wv_{layer}"])

        q = q.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        q = rms_norm(self.rope(q, pos=pos))
        k = rms_norm(self.rope(k, pos=pos))

        # the part till above it stays same even if we want to compute only the last row of q, k, v (we call the func with L = 1, for just the next token); L can also be > 1, for example, if we have a initial prompt, we can process it completely in 1 forward pass

        # K-V cache updates
        if cache is not None:
            cache[layer]["k"][:, :, pos:pos+L,:] = k # first 2 : are for Batch_size and num_heads, pos to pos+L is the pos of this subset of tokens in the seq, last : is for head_dim
            cache[layer]["v"][:, :, pos:pos+L,:] = v
            k = cache[layer]["k"][:, :, :pos+L, :] # this assigns the entire cached matrix k for the seq from beginning to pos + L (current pos)
            v = cache[layer]["v"][:, :, :pos+L, :]

        out = self_attn(q, k, v, mask)
        out = out.transpose(1, 2).reshape(B, L, d)
        return linear(out, self.params[f"wp_{layer}"]) # final projection after multi-head attn oprn


    def mlp(self, x, layer):

        return linear(silu(linear(x, self.params[f"w1_{layer}"])), self.params[f"w2_{layer}"])


    def transformer_block(self, x, layer, mask, pos=0, cache=None):

        x = x + self.multihead_attn(rms_norm(x), layer, mask, pos=pos, cache=cache)
        return x + self.mlp(rms_norm(x), layer)

    def __init__(self, config, init_weights=True):

        # hyperparams
        d = config["depth"]*config["aspect_ratio"]
        mlp_dim = d * config["mlp_multiple"]
        self.num_layers = config["depth"]
        self.head_dim = config["head_dim"]
        self.num_heads = d // self.head_dim
        self.seq_len = config["seq_len"]
        self.vocab_size = config["vocab_size"]
        self.rope_theta = config["rope_theta"]
        torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}

        dtype = config["dtype"]
        self.dtype = torch_types[dtype] if isinstance(dtype, str) else dtype

        # self.num_gpus = config["num_gpus"]
        # self.batch_size = config["batch_size"]
        # self.token_multiple = config["token_multiple"]
        # self.lr = config["lr"]
        # self.data_mix = config["data_mix"]
        # self.tokenizer = config["tokenizer"]

        if d % self.head_dim or self.head_dim % 2:
            raise ValueError("Embedding width must divide into heads with an even head_dim.")

        # Avoid initializing a second full random model when loading a checkpoint.
        def allocate(rows, cols, scale):
            if init_weights:
                return torch.randn(rows, cols) * scale
            return torch.empty(rows, cols)

        # trainable params
        self.params = {}

        scale, scale_mlp = math.sqrt(2/d), math.sqrt(2/mlp_dim)
        self.params["embedding"] = allocate(self.vocab_size, d, scale)
        self.params["output"] = allocate(d, self.vocab_size, scale)

        for i in range(self.num_layers):
            self.params[f"wq_{i}"] = allocate(d, d, scale)
            self.params[f"wk_{i}"] = allocate(d, d, scale)
            self.params[f"wv_{i}"] = allocate(d, d, scale)
            self.params[f"wp_{i}"] = allocate(d, d, scale)
            self.params[f"w1_{i}"] = allocate(d, mlp_dim, scale)
            self.params[f"w2_{i}"] = allocate(mlp_dim, d, scale_mlp)

        # params/tensors which we don't want to optimize are stored in buffers
        self.buffers = {}
        mask = torch.full((self.seq_len, self.seq_len ), -float("inf"))
        self.buffers["mask"] = torch.triu(mask, diagonal=1)

        # RoPE
        rope_freqs = self.rope_theta ** (-2 * torch.arange(self.head_dim//2)/self.head_dim)
        theta = torch.outer(torch.arange(self.seq_len), rope_freqs)
        cos, sin = torch.cos(theta), torch.sin(theta)
        self.buffers["rope1"] = torch.stack([cos, cos], dim=-1).reshape(self.seq_len, self.head_dim)
        self.buffers["rope2"] = torch.stack([-sin, sin], dim=-1).reshape(self.seq_len, self.head_dim)

        for k,v in self.buffers.items():
            self.buffers[k] = v.to(dtype=self.dtype)


    def __call__(self, tokens, pos=0, cache = None):
        if tokens.ndim != 2 or tokens.shape[1] == 0:
            raise ValueError("tokens must have shape [batch, nonempty sequence].")
        if pos < 0 or pos + tokens.shape[1] > self.seq_len:
            raise ValueError("Requested positions exceed the configured context length.")
        if pos and cache is None:
            raise ValueError("A nonzero position requires the preceding KV cache.")
        x = rms_norm(embedding(tokens, self.params["embedding"], self.dtype))
        mask = self.buffers["mask"][pos:pos+tokens.shape[1], :pos+tokens.shape[1]]
        for i in range(self.num_layers):
            x = self.transformer_block(x, i, mask, pos=pos, cache=cache)
        return linear(rms_norm(x), self.params["output"])

    @torch.inference_mode()
    def generate(self, prompts, max_tokens=500, temp=0.7, eos=None):
     
        if prompts.ndim != 2 or prompts.shape[0] == 0 or prompts.shape[1] == 0:
            raise ValueError("Use a nonempty [batch, prompt_length] tensor of token ids.")
        if prompts.dtype != torch.long:
            raise ValueError("prompts must use torch.long token ids")
        if eos is not None and not 0 <= eos < self.vocab_size:
            raise ValueError("eos must be a valid token id or None")
        if prompts.device != self.params["embedding"].device:
            raise ValueError("Prompts and model weights must be on the same device.")

        B, prompt_length = prompts.shape
        capacity = prompt_length + max_tokens

        if capacity > self.seq_len:
            raise ValueError(f"Prompt + generation budget ({capacity}) exceeds context ({self.seq_len}).")
        if max_tokens == 0:
            return prompts.clone()

        tokens = torch.empty((B, capacity), device=prompts.device, dtype=torch.long)
        tokens[:, :prompt_length] = prompts
        cache = [{
            "k": torch.empty(B, self.num_heads, capacity, self.head_dim,
                             device=prompts.device, dtype=self.dtype),
            "v": torch.empty(B, self.num_heads, capacity, self.head_dim,
                             device=prompts.device, dtype=self.dtype)
        } for _ in range(self.num_layers)]
        
        finished = torch.zeros(B, device=prompts.device, dtype=torch.bool)
        start, end = 0, prompt_length

        while end < capacity:
            logits = self(tokens[:, start:end], pos=start, cache=cache)[:, -1]
            probs = softmax(logits.float() / temp)
            next_tokens = torch.multinomial(probs, 1).squeeze(-1)
            if eos is not None:
                # Freeze finished sequences at EOS while other rows continue.
                next_tokens = torch.where(finished, torch.full_like(next_tokens, eos), next_tokens)
                finished |= next_tokens.eq(eos)
            tokens[:, end] = next_tokens
            start, end = end, end + 1
            if eos is not None and finished.all().item():
                break
        return tokens[:, :end]
    

    def save(self, filename):

        torch.save(self.params, filename)


    def load(self, filename):

        state = torch.load(filename, map_location="cpu", weights_only=True)
        if not isinstance(state, dict) or set(state) != set(self.params):
            raise ValueError("Checkpoint parameter names do not match this architecture.")
        for name, expected in self.params.items():
            value = state[name]
            if not isinstance(value, torch.Tensor) or value.shape != expected.shape:
                raise ValueError(f"Wrong tensor/shape for {name}; expected {tuple(expected.shape)}.")
        self.params = state

    def to(self, device, dtype=None, parameter_dtype=None):
        self.dtype = self.dtype if dtype is None else dtype
        parameter_dtype = self.dtype if parameter_dtype is None else parameter_dtype
        # need to call this before constructing the optimizer, as moving can replace tensor objects
        self.params = {k: v.to(device=device, dtype=parameter_dtype) for k, v in self.params.items()}
        self.buffers = {k: v.to(device=device, dtype=self.dtype) for k, v in self.buffers.items()}
        return self




### Single Device Training

In [48]:
RUN_TRAINING = False
TRAIN_FROM_CHECKPOINT = False

if TRAIN_FROM_CHECKPOINT:
    train_config = json.loads((MODEL_DIR / "config.d30.json").read_text())
else:
    train_config = {
        "depth": 6, "aspect_ratio": 64, "mlp_multiple": 4,
        "head_dim": 128, "vocab_size": 32768, "rope_theta": 10000,
    }

train_config.update({
    "dtype": torch.float32,      # Training compute; CUDA BF16 is optional if supported.
    "seq_len": 256, "batch_size": 1,
    "total_steps": 100, "lr": 3e-4, "weight_decay": 0.025,
    "tokenizer": str(MODEL_DIR / "tokenizer_50M.bpe"),
    "data_mix": {str(DATA_DIR / "fineweb-edu-10BT.shuffle.bin"): 1.0},
})
# adding matching local datasets with weights, e.g.:
# train_config["data_mix"] = {str(DATA_DIR / "fineweb-edu-10BT.shuffle.bin"): 0.875,
#                             str(DATA_DIR / "smoltalk.shuffle.bin"): 0.125}

In [49]:
def cross_entropy_loss(logits, y):

    logits = logits - logits.max(dim=-1, keepdim=True)[0]
    loss = -logits.take_along_dim(y.unsqueeze(-1), dim=-1)
    loss += logits.exp().sum(dim=-1, keepdim=True).log()
    return loss.mean()


class Adam:
    def __init__(self, params, lr_schedule, betas = (0.9, 0.95), eps=1e-5, weight_decay=0.0):

        self.schedule = lr_schedule
        self.betas = betas
        self.params = params
        self.u = {k: torch.zeros_like(v) for k, v in params.items()}
        self.v = {k: torch.zeros_like(v) for k, v in params.items()}
        self.t = 1
        self.eps = eps
        self.weight_decay = weight_decay

        for p in self.params.values():
            p.requires_grad_(True)


    def step(self):

        with torch.no_grad():
            for k, p in self.params.items():
                self.u[k] = self.betas[0] * self.u[k] + (1 - self.betas[0]) * p.grad
                self.v[k] = self.betas[1] * self.v[k] + (1 - self.betas[1]) * p.grad**2
                u_hat = self.u[k] / (1 - self.betas[0]**self.t)
                v_hat = self.v[k] / (1 - self.betas[1]**self.t)

                epoch_lr = self.schedule.get_lr(self.t)
                p *= (1 - epoch_lr*self.weight_decay)
                p -= epoch_lr * u_hat / (v_hat.sqrt() + self.eps)

                p.grad.zero_()

            self.t += 1


class LRSchedule:
    def __init__(self, total_steps, lr=1e-3, warmup_steps=50, decay_ratio=0.4, min_frac=0.1):

        self.total_steps = total_steps
        self.lr = lr
        self.warmup_steps = warmup_steps
        self.decay_steps = round(decay_ratio*total_steps)
        self.min_frac = min_frac


    def get_lr(self, step):

        if step < self.warmup_steps:
            p = step / self.warmup_steps
            return (1-p)*self.min_frac*self.lr + p*self.lr
        elif step > (self.total_steps - self.decay_steps):
            p = (step - (self.total_steps - self.decay_steps)) / self.decay_steps
            return (1-p)*self.lr + p*self.min_frac*self.lr
        else:
            return self.lr

In [50]:
from array import array
import random
import sys
import os
import time

def train_llm(config, initial_checkpoint=None):

    tokenizer = BPE(config["tokenizer"])
    # if len(tokenizer.vocab) != config["vocab_size"]:
    #     raise ValueError("Tokenizer vocabulary and model config disagree")
    if config["batch_size"] < 1 or config["total_steps"] < 1:
        raise ValueError("batch_size and total_steps must be positive")

    if config["dtype"] == torch.bfloat16 and (DEVICE.type != "cuda" or not torch.cuda.is_bf16_supported()):
        raise ValueError("Use float32 training unless your CUDA GPU supports BF16")

    llm = LLM(config, init_weights=initial_checkpoint is None)
    
    if initial_checkpoint is not None:
        llm.load(initial_checkpoint)

    llm.to(DEVICE, config["dtype"], parameter_dtype=torch.float32)

    config["total_params"] = sum(p.numel() for p in llm.params.values())
    config["total_tokens"] = config["total_params"]*config["token_multiple"]
    config["total_steps"] = config["total_tokens"] // (config["batch_size"]*config["seq_len"])
    
    schedule = LRSchedule(config["total_steps"], lr=config["lr"])
    optimizer = Adam(llm.params, schedule, weight_decay=config["weight_decay"])
    if log:
        run = wandb.init(project="llm_speedrun", config=config)

    # creating tuples of (filename, batch_item_offset) for every batch
    batch_items = []
    for k, p in config["data_mix"].items():
        batch_items += [(k, i*(config["seq_len"]+1)*2) for i in range(round(p*config["batch_size"]))] # tuples of the form (0, 0), (0, 2*(seq_len + 1), ..)
    file_offsets = {k:0 for k in config['data_mix']}
    n_tokens = 0

    while optimizer.t < config["total_steps"]:
        # load tokens for that batch
        start_time = time.perf_counter()
        tokens = []

        for filename, offset in batch_items:
            with open(filename, "rb") as f:
                f.seek(file_offsets[filename] + offset)
                tokens += array("H", f.read(2*(config["seq_len"]+1))).tolist() # the H converts tokens to 16b precision
        for k, p in config["data_mix"].items():
            read_size = round(p*config["batch_size"]*(config["seq_len"]+1)*2)
            file_offsets[k] += read_size
            if (file_offsets[k] + read_size >= os.path.getsize(k)):
                file_offsets[k] = 0

        tokens = torch.tensor(tokens).reshape(config["batch_size"], config["seq_len"]+1)
        text = tokenizer.decode(tokens[:, 1:].flatten().tolist()) # target seq for each input seq, for evaluating the quality of the llm using bits per bytes

        # run and take optimizer step
        tokens = tokens.cuda()
        logits = llm(tokens[:, :-1]).float() # llm will probably output in 16b, we convert the logits to 32b float before computing loss, else we will lose precision
        loss = cross_entropy_loss(logits, tokens[:, 1:])
        loss.backward()
        optimizer.step()

        # logging
        bpb = (loss.item()*local_batch_size*config["seq_len"]) / (len(text)*math.log(2))
        n_tokens+= config["batch_size"]*config["seq_len"]
        tokens_per_sec = (config["batch_size"]*config["seq_len"]) / (time.perf_counter() - start_time)
        if log:
            run.log({"step": optimizer.t,
                     "tokens": n_tokens,
                     "loss": loss.item(),
                     "bpb": bpb,
                     "lr": optimizer.schedule.get_lr(optimizer.t),
                     "tok_per_sec": tokens_per_sec})


        return llm

        

In [ ]:
if RUN_TRAINING:
    initial_checkpoint = MODEL_DIR / "llm.d30.pt" if TRAIN_FROM_CHECKPOINT else None
    trained_llm = train_llm(train_config, initial_checkpoint=initial_checkpoint)
    trained_bpe = BPE(train_config["tokenizer"])

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    trained_llm.save(OUTPUT_DIR / "model.pt")
    trained_bpe.save(OUTPUT_DIR / "tokenizer.bpe")
    saved_config = dict(train_config)
    saved_config["dtype"] = str(saved_config["dtype"]).removeprefix("torch.")
    saved_config["tokenizer"] = "tokenizer.bpe"
    saved_config["filename"] = "model.pt"
    (OUTPUT_DIR / "config.json").write_text(json.dumps(saved_config, indent=2))
    print("Saved model, tokenizer, and config to:", OUTPUT_DIR.resolve())
else:
    print("Training disabled. Edit cell 13 and set RUN_TRAINING=True when ready.")

In [52]:
# generate from trained model

if RUN_TRAINING:
    prompt = "Once upon a time"
    prompts = torch.tensor([trained_bpe.encode(prompt)], device=DEVICE, dtype=torch.long)
    budget = min(80, trained_llm.seq_len - prompts.shape[1])
    tokens = trained_llm.generate(prompts, max_tokens=budget, temp=0.7, eos=None)
    print(trained_bpe.decode(tokens.tolist()[0]))

In [53]:
# config = json.loads((OUTPUT_DIR / "config.json").read_text())
# config["dtype"] = DTYPE
# trained_llm = LLM(config, init_weights=False)
# trained_llm.load(OUTPUT_DIR / "model.pt")
# trained_llm.to(DEVICE, DTYPE)
# trained_bpe = BPE(OUTPUT_DIR / "tokenizer.bpe")

### Pretrained inference

In [54]:
if RUN_INFERENCE:
    for name in ("llm.d30.pt", "config.d30.json", "tokenizer_50M.bpe"):
        if not (MODEL_DIR / name).is_file():
            raise FileNotFoundError(MODEL_DIR / name)

In [56]:
if RUN_INFERENCE:
    config = json.loads((MODEL_DIR / "config.d30.json").read_text())
    config["dtype"] = DTYPE
    bpe = BPE(MODEL_DIR / "tokenizer_50M.bpe")
    # if len(bpe.vocab) != config["vocab_size"]:
    #     raise ValueError("Tokenizer vocabulary and model config disagree.")
    llm = LLM(config, init_weights=False)
    llm.load(MODEL_DIR / "llm.d30.pt")
    llm.to(DEVICE, DTYPE)

In [57]:
if RUN_INFERENCE:
    prompt = "<DOCUMENT><USER>Why is the sky blue?</USER><ASSISTANT>"
    prompts = torch.tensor([bpe.encode(prompt)], device=DEVICE, dtype=torch.long)
    eos = bpe.vocab.index("</ASSISTANT>")
    budget = min(500, llm.seq_len - prompts.shape[1])
    tokens = llm.generate(prompts, max_tokens=budget, temp=0.7, eos=eos)
    print(bpe.decode(tokens.tolist()[0]))

<DOCUMENT><USER>Why is the sky blue?</USER><ASSISTANT>The sky is blue because of the presence of a type of molecule called a dispersion type of light scattering molecule. Inside a typical human eye, these molecules scatter blue light more than other types of light, making the sky appear blue.</ASSISTANT>


In [62]:
if RUN_INFERENCE:
    prompt = "<DOCUMENT><USER>Write a poem about the ocean.</USER><ASSISTANT>"
    prompts = torch.tensor([bpe.encode(prompt)], device=DEVICE, dtype=torch.long)
    eos = bpe.vocab.index("</ASSISTANT>")
    budget = min(500, llm.seq_len - prompts.shape[1])
    tokens = llm.generate(prompts, max_tokens=budget, temp=0.7, eos=eos)
    print(bpe.decode(tokens.tolist()[0]))

<DOCUMENT><USER>Write a poem about the ocean.</USER><ASSISTANT>The ocean's vast and mysterious,
A canvas of endless blue,
A canvas of endless surf.

Beneath its waves, a secret lies,
A world of wonder and magic.

Curvy waves, a labyrinth to trace,
A maze of currents and ripples.

A place where the ancient ones left their scars,
A place where the shipwrecks of the years remain.

The ocean's ecosystem's heart,
A place of life and death,
A world of sound and silence.

The ocean's whispers, a soft caress,
A gentle touch to the land.

The ocean's magic, a symphony to hear,
A symphony of the waves and the trees.

The ocean's secrets, a mystery to unravel,
A mystery of the past and the future.

The ocean's sway, a rhythm to know,
A rhythm of the sky and the land.

The ocean's song, a melody to sing,
A song of the past and the present.

The ocean's secrets, a mystery to love,
A mystery of the world's beauty.

The ocean's name, a name to cherish and defend,
A name that whispers secrets and stor

In [65]:
if RUN_INFERENCE:
    prompt = "<DOCUMENT><USER>Write a poem about an LLM.</USER><ASSISTANT>"
    prompts = torch.tensor([bpe.encode(prompt)], device=DEVICE, dtype=torch.long)
    eos = bpe.vocab.index("</ASSISTANT>")
    budget = min(500, llm.seq_len - prompts.shape[1])
    tokens = llm.generate(prompts, max_tokens=budget, temp=0.7, eos=eos)
    print(bpe.decode(tokens.tolist()[0]))

<DOCUMENT><USER>Write a poem about an LLM.</USER><ASSISTANT>In the dimly lit halls of the Golden Keys,
The LLM's gaze is a beacon of light,
Her words are like a revelation of truth,
Guiding the minds of the young and the old.

The LLM's eyes shine brightly, a beacon of light,
As she sips from the cup of knowledge,
In the nighttime hours, a soft glow appears,
A soft glow from the heart of the LLM,
A beacon of light in the night.

The LLM's words are a source of wonder and awe,
She speaks with a voice as clear as the sea,
As she teaches the young and the old,
Her words are a testament to her wisdom,
A testament to the LLM's heart and her love.

The LLM's eyes hold secrets and stories of the past,
Her words are a treasure trove of wisdom,
As she guides the minds of the young and the old,
Her words are a guide to the depths of the past,
A guide to the depths of the past, her words are a guide.

The LLM's words are like a lantern in the night,
A beacon of light in the darkness,
A source of 